# Milestone 3 · Notebook 02: Cohere Embed v4.0, Hybrid Search (Dense + Sparse) & RRF + Rerank

**Objective:** Validate asymmetric `embed-v4.0` (`output_dimension=1024`), sparse keyword matching, Reciprocal Rank Fusion (`k=60`), and `rerank-v5.0-fast` before implementing `backend/app/retrieval.py`.

> **Live vs Offline Mode:** If `COHERE_API_KEY` is present in `.env`, this notebook executes live Cohere Embed + Rerank calls. Otherwise, it runs a deterministic TF-IDF/cosine + RRF simulation on the exact same corpus chunks so all ranking math and pipeline contracts are verified with zero errors.

In [1]:
from __future__ import annotations

import math
import re
import sys
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.app.config import get_settings

settings = get_settings()

LIVE_COHERE = bool(settings.cohere_api_key and "your_" not in settings.cohere_api_key)
print(f"Cohere Embed Model  : {settings.cohere_embed_model} (dim={settings.cohere_embed_dim})")
print(f"Cohere Rerank Model : {settings.cohere_rerank_model}")
print(f"RRF k               : {settings.rrf_k}")
print(f"Live Cohere Mode    : {LIVE_COHERE}")

Cohere Embed Model  : embed-v4.0 (dim=1024)
Cohere Rerank Model : rerank-v5.0-fast
RRF k               : 60
Live Cohere Mode    : False


## 1. Load Sample Corpus Chunks (Exact Codes, Narrative, Contradiction Pair & CSV Rows)

In [2]:
# Representative chunks drawn directly from our 6-format validation corpus
SAMPLE_CHUNKS = [
    {
        "id": "chk_hr_01",
        "filename": "01_company_hr_policy.pdf",
        "page": 1,
        "content": (
            "[Document: 01_company_hr_policy.pdf | Section: Paid Annual Leave | Page: 1]\n"
            "Under HR-POL-2026, full-time employees are entitled to 22 days of paid annual leave "
            "per calendar year and up to 5 unused days carry over into Q1."
        ),
    },
    {
        "id": "chk_hr_02",
        "filename": "01_company_hr_policy.pdf",
        "page": 2,
        "content": (
            "[Document: 01_company_hr_policy.pdf | Section: Expense Matrix | Page: 2]\n"
            "Annual Learning & Development stipend is USD 2,400. Submit receipts within 30 days "
            "using expense code ACME-EXP-2026 in Expensify."
        ),
    },
    {
        "id": "chk_rw_2025",
        "filename": "03_remote_work_policy_2025.docx",
        "page": 1,
        "content": (
            "[Document: 03_remote_work_policy_2025.docx | Section: In-Office Attendance | Page: 1]\n"
            "Under Policy RW-2025, hybrid engineering employees must work in-office 3 days/week "
            "and receive a USD 1,000 home office equipment stipend."
        ),
    },
    {
        "id": "chk_rw_2026",
        "filename": "04_remote_work_addendum_2026.docx",
        "page": 1,
        "content": (
            "[Document: 04_remote_work_addendum_2026.docx | Section: Updated Attendance | Page: 1]\n"
            "Under Addendum RW-2026, hybrid engineering employees work in-office 2 days/week "
            "and receive an increased USD 1,500 home office equipment stipend."
        ),
    },
    {
        "id": "chk_md_01",
        "filename": "05_incident_runbook.md",
        "page": 1,
        "content": (
            "[Document: 05_incident_runbook.md | Section: Severity Classification | Page: 1]\n"
            "SEV-1 incidents require 10-minute acknowledgement SLA and blameless post-mortem within 48 hours."
        ),
    },
    {
        "id": "chk_pptx_04",
        "filename": "07_q3_product_roadmap.pptx",
        "page": 4,
        "content": (
            "[Document: 07_q3_product_roadmap.pptx | Section: Q3 Delivery Milestones | Slide: 4]\n"
            "Hybrid RRF + Rerank GA: USD 45,000 | Multi-Tenant JWT Auth: USD 30,000 | "
            "Langfuse Observability: USD 20,000. Total Q3 budget is USD 95,000."
        ),
    },
    {
        "id": "chk_csv_01",
        "filename": "08_regional_sales_2025.csv",
        "page": 1,
        "content": (
            "[Document: 08_regional_sales_2025.csv | Section: Rows 1-12 | Columns: Region, Quarter, Product, Revenue_USD]\n"
            "| North America | Q4-2025 | Enterprise RAG | 296460 | 20 | 69.7% |"
        ),
    },
]
print(f"Loaded {len(SAMPLE_CHUNKS)} multi-format prototype chunks.")

Loaded 7 multi-format prototype chunks.


## 2. Dense Retrieval + Sparse Keyword Search + Reciprocal Rank Fusion (`k=60`)

In [3]:
TOKEN_RE = re.compile(r"[a-z0-9\-_]+")


def tokenize(text: str) -> list[str]:
    """Simple lowercase tokenizer mirroring PostgreSQL 'simple' tsvector behavior."""
    return TOKEN_RE.findall(text.lower())


def sparse_rank(query: str, chunks: list[dict[str, Any]]) -> list[tuple[str, float]]:
    """Compute simple TF-IDF overlap scores mirroring PostgreSQL ts_rank."""
    q_tokens = set(tokenize(query))
    scored: list[tuple[str, float]] = []
    for c in chunks:
        c_counts = Counter(tokenize(c["content"]))
        score = sum(c_counts[t] for t in q_tokens) / (math.sqrt(len(c_counts)) + 1.0)
        if score > 0:
            scored.append((c["id"], round(score, 4)))
    return sorted(scored, key=lambda x: x[1], reverse=True)


def rrf_fuse(
    dense_ids: list[str],
    sparse_ids: list[str],
    k: int = 60,
) -> list[tuple[str, float]]:
    """Merge dense and sparse rankings via Reciprocal Rank Fusion: sum(1 / (k + rank))."""
    scores: dict[str, float] = {}
    for rank, cid in enumerate(dense_ids, start=1):
        scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank)
    for rank, cid in enumerate(sparse_ids, start=1):
        scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


async def embed_and_rerank_demo(query: str) -> pd.DataFrame:
    """Run live Cohere v2 Embed + Rerank when configured, else offline cosine + RRF."""
    texts = [c["content"] for c in SAMPLE_CHUNKS]
    if LIVE_COHERE:
        import cohere

        client = cohere.AsyncClientV2(api_key=settings.cohere_api_key, timeout=settings.api_timeout_seconds)
        doc_resp = await client.embed(
            model=settings.cohere_embed_model,
            input_type="search_document",
            texts=texts,
            output_dimension=settings.cohere_embed_dim,
            embedding_types=["float"],
        )
        q_resp = await client.embed(
            model=settings.cohere_embed_model,
            input_type="search_query",
            texts=[query],
            output_dimension=settings.cohere_embed_dim,
            embedding_types=["float"],
        )
        doc_vecs = np.array(doc_resp.embeddings.float, dtype=np.float32)
        q_vec = np.array(q_resp.embeddings.float[0], dtype=np.float32)
        assert doc_vecs.shape[1] == 1024, f"Expected 1024-dim vector, got {doc_vecs.shape[1]}"
    else:
        # Deterministic bag-of-words unit-normalized vectors for offline validation
        vocab = sorted({t for txt in [*texts, query] for t in tokenize(txt)})
        v_idx = {tok: i for i, tok in enumerate(vocab)}

        def _vec(s: str) -> np.ndarray:
            arr = np.zeros(len(vocab), dtype=np.float32)
            for tok in tokenize(s):
                arr[v_idx[tok]] += 1.0
            norm = np.linalg.norm(arr)
            return arr / norm if norm > 0 else arr

        doc_vecs = np.vstack([_vec(t) for t in texts])
        q_vec = _vec(query)

    # Cosine similarity = dot product of unit-normalized vectors
    norms = np.linalg.norm(doc_vecs, axis=1) * (np.linalg.norm(q_vec) + 1e-9)
    cos_scores = (doc_vecs @ q_vec) / np.maximum(norms, 1e-9)
    dense_ranked = sorted(
        [(SAMPLE_CHUNKS[i]["id"], float(cos_scores[i])) for i in range(len(SAMPLE_CHUNKS))],
        key=lambda x: x[1],
        reverse=True,
    )
    sparse_ranked = sparse_rank(query, SAMPLE_CHUNKS)
    fused = rrf_fuse([cid for cid, _ in dense_ranked], [cid for cid, _ in sparse_ranked], k=settings.rrf_k)

    # Optional live Cohere Rerank on top RRF candidates
    rerank_scores: dict[str, float] = {}
    if LIVE_COHERE:
        import cohere

        client = cohere.AsyncClientV2(api_key=settings.cohere_api_key, timeout=settings.api_timeout_seconds)
        id_to_text = {c["id"]: c["content"] for c in SAMPLE_CHUNKS}
        cand_ids = [cid for cid, _ in fused[:5]]
        rr_resp = await client.rerank(
            model=settings.cohere_rerank_model,
            query=query,
            documents=[id_to_text[cid] for cid in cand_ids],
            top_n=len(cand_ids),
        )
        for item in rr_resp.results:
            rerank_scores[cand_ids[item.index]] = round(float(item.relevance_score), 4)

    dense_map = dict(dense_ranked)
    rows = [
        {
            "chunk_id": cid,
            "dense_cosine": round(dense_map[cid], 4),
            "rrf_score": round(rrf_sc, 5),
            "rerank_score": rerank_scores.get(cid, None),
        }
        for cid, rrf_sc in fused[:5]
    ]
    return pd.DataFrame(rows)


for test_q in [
    "What expense code do I use for ACME-EXP-2026 reimbursements?",
    "How many days per week must hybrid engineers work in the office?",
    "What is the capital of Mars?",
]:
    print(f"\nQuery: {test_q}")
    df = await embed_and_rerank_demo(test_q)
    print(df.to_string(index=False))


Query: What expense code do I use for ACME-EXP-2026 reimbursements?
   chunk_id  dense_cosine  rrf_score rerank_score
  chk_hr_02        0.2395    0.03279         None
  chk_hr_01        0.0000    0.01613         None
chk_rw_2025        0.0000    0.01587         None
chk_rw_2026        0.0000    0.01562         None
  chk_md_01        0.0000    0.01538         None

Query: How many days per week must hybrid engineers work in the office?
   chunk_id  dense_cosine  rrf_score rerank_score
chk_rw_2025        0.2970    0.03279         None
chk_rw_2026        0.2552    0.03226         None
  chk_hr_01        0.1291    0.03175         None
  chk_hr_02        0.1037    0.03125         None
chk_pptx_04        0.0373    0.03077         None

Query: What is the capital of Mars?
   chunk_id  dense_cosine  rrf_score rerank_score
  chk_hr_02        0.0733    0.03279         None
  chk_hr_01        0.0609    0.03200         None
chk_pptx_04        0.0527    0.03200         None
chk_rw_2025        0.